# 05 — Build FAISS Index for the Fine-Tuned Embedder

This notebook re-embeds the full corpus with the fine-tuned `e5-large-tr-legal` model and writes a FAISS index that the retriever can pick up at evaluation time.

The base `intfloat/multilingual-e5-large` index is already shipped with the Kaggle dataset; the FT variant is built here because it is too slow to produce on the local 4 GB GPU.

**Output.**
- `faiss_data_models_e5-large-tr-legal.index` (~164 MB)
- `embeddings_data_models_e5-large-tr-legal.npy` (~160 MB)

**Note on file naming.** `Retriever` sanitizes the embed-model path (`/` → `_`) and looks up `faiss_<sanitized>.index`. The build script writes exactly that filename — no rename is needed.

**Wall time.** ~10–12 minutes on a single T4 with batch size 32.

In [ ]:
%%capture
!pip install -q sentence-transformers faiss-cpu rank-bm25

In [ ]:
import os, shutil
from pathlib import Path

INPUT = Path('/kaggle/input/datasets/hasanemreusta/turkish-legal-rag-system')
WORK = Path('/kaggle/working/turkish-legal-rag')
WORK.mkdir(exist_ok=True, parents=True)

if not (WORK / 'src').exists():
    shutil.copytree(INPUT / 'src', WORK / 'src')
if not (WORK / 'data').exists():
    os.symlink(INPUT / 'data', WORK / 'data')

os.chdir(WORK)
print('CWD:', os.getcwd())
print('FT model exists:', (WORK/'data/models/e5-large-tr-legal').exists())
print('Meta exists:', (WORK/'data/index_full/meta.jsonl').exists())

# Count records in meta
import json
n = sum(1 for _ in open(WORK/'data/index_full/meta.jsonl', encoding='utf-8'))
print(f'Meta records: {n}')

## Build the dense index

The shipped `meta.jsonl` is in the same schema as the original corpus (`doc_id`, `kanun_full`, `madde_no`, `baslik`, `text`), so we can pass it directly as `--corpus`. The BM25 component is skipped since only the dense index needs rebuilding.

In [ ]:
!python -m src.retrieval.build_index \
  --corpus data/index_full/meta.jsonl \
  --output-dir /kaggle/working/ft_index \
  --embed-model data/models/e5-large-tr-legal \
  --skip-bm25 \
  --verbose

## Verify outputs

The retriever resolves the FAISS filename from the embed-model path. Confirm that the produced files match the expected names.

In [ ]:
import os
out = Path('/kaggle/working/ft_index')
for f in out.iterdir():
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'{f.name}  ({size_mb:.1f} MB)')

## Distribute

Download the two files from the Kaggle Output panel and copy them into `data/index_full/` locally, or upload them as a new version of the `turkish-legal-rag-system` Kaggle dataset so that downstream evaluation notebooks see them under `/kaggle/input/.../data/index_full/`.